# Queen Editor — Colab kurulumu

Drive'ı bağlar → repoyu klonlar → **ComfyUI'yi kurar** (21 custom node) → **seçtiğin üreticilerin
modellerini indirir** → **Flask** arayüzünü **cloudflared** linkiyle açar. Projeler
`MyDrive/queenEditor/<proje>/` altında.

## Kullanım
1. Bu `queeneditor.ipynb`'yi Colab'a yükle (**File → Upload notebook**).
2. **🔑 Secrets:** `GITHUB_TOKEN` (fine-grained, yalnız bu repo, `Contents: read`); `HF_TOKEN`
   (fine-grained, yalnız CONFIG'deki `HF_MIRROR` reposu, okuma + yazma — Civitai dosyaları önce
   oradan iner, yenileri oraya yüklenir); `CIVITAI_COOKIE` yalnız aynada henüz olmayan bir dosya
   için (civitai.red → F12 → Application → Cookies → `__Secure-civ-token`, ~30 günde bir yenilenir);
   video için `XAI_API_KEY` (video prompt'unu yazan dil modeli).
3. CONFIG'de üreticileri ve modellerini işaretle.
4. **Runtime → Run all** → Drive izni ver → ~10-60 dk → en alttaki linke gir.

In [ ]:
# === Cell timer ===
import time
from IPython import get_ipython

def _cell_began(*_):
    global _CELL_START
    _CELL_START = time.perf_counter()

def cell_elapsed():
    minutes, seconds = divmod(round(time.perf_counter() - _CELL_START), 60)
    return f"{minutes} dk {seconds} sn" if minutes else f"{seconds} sn"

def _cell_ended(*_):
    print(f"⏱️ Hücre {cell_elapsed()} sürdü")

_events = get_ipython().events
for _event, _hook in (("pre_run_cell", _cell_began), ("post_run_cell", _cell_ended)):
    for _old in [h for h in _events.callbacks[_event] if h.__name__ == _hook.__name__]:
        _events.unregister(_event, _old)
    _events.register(_event, _hook)
_cell_began()

# === CONFIG ===

#@markdown ### Üreticiler
#@markdown En az birini işaretle. Video ~39 GiB · ses ~9 GiB.
INSTALL_PHOTO = False  #@param {type:"boolean"}
INSTALL_VIDEO = False  #@param {type:"boolean"}
INSTALL_AUDIO = False  #@param {type:"boolean"}

#@markdown ---
#@markdown ### Fotoğraf modelleri
PHOTO_NOVA3DCG = False  #@param {type:"boolean"}
PHOTO_DASIWA = False  #@param {type:"boolean"}

#@markdown ---
#@markdown ### Video modelleri
VIDEO_WAN = False  #@param {type:"boolean"}
VIDEO_H3 = False  #@param {type:"boolean"}

assert INSTALL_PHOTO or INSTALL_VIDEO or INSTALL_AUDIO, (
    "❌ Hiçbir üretici seçilmedi — yukarıdaki kutulardan en az birini işaretle "
    "(INSTALL_PHOTO / INSTALL_VIDEO / INSTALL_AUDIO) ve hücreyi tekrar çalıştır."
)
assert not INSTALL_PHOTO or PHOTO_NOVA3DCG or PHOTO_DASIWA, (
    "❌ Fotoğraf seçildi ama hiç model işaretlenmedi — yukarıdaki model kutularından "
    "en az birini işaretle ve hücreyi tekrar çalıştır."
)
assert not INSTALL_VIDEO or VIDEO_WAN or VIDEO_H3, (
    "❌ Video seçildi ama model işaretlenmedi — VIDEO_WAN ya da VIDEO_H3'ü işaretle."
)
assert not (VIDEO_WAN and VIDEO_H3), (
    "❌ WAN ve H3 aynı oturumda kurulmaz — video modellerinden yalnız birini işaretle."
)
VIDEO_MODEL = ("wan" if VIDEO_WAN else "h3") if INSTALL_VIDEO else ""

from google.colab import userdata

try:
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    GITHUB_TOKEN = ""

BRANCH       = "feat/queen-editor-v7"
REPO         = "AltanBaysal/Internal-tools"
CLONE_DIR    = "/content/Internal-tools"
APP_DIR      = f"{CLONE_DIR}/queen-editor"
APP_PORT     = 8000
DRIVE_FOLDER = "queenEditor"
HF_MIRROR    = "Test468735/queen-editor-models"

# === ComfyUI ===
COMFY_PORT  = 8188
COMFY_ROOT  = "/content/ComfyUI"
COMFY_LOG   = "/content/comfyui.log"
COMFYUI_URL = f"http://127.0.0.1:{COMFY_PORT}"

# === Secrets ===
try:
    COOKIE_VALUE = userdata.get("CIVITAI_COOKIE")
except Exception:
    COOKIE_VALUE = ""

try:
    XAI_API_KEY = (userdata.get("XAI_API_KEY") or "").strip()
except Exception:
    XAI_API_KEY = ""

assert GITHUB_TOKEN, (
    "❌ GITHUB_TOKEN yok — Colab solundaki 🔑 Secrets panelinden 'GITHUB_TOKEN' adıyla ekle "
    "ve bu notebook'a erişimi aç (fine-grained, yalnız bu repo, Contents: read)."
)

# === GPU ===
import subprocess as _sp
try:
    _gpu = _sp.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                   capture_output=True, text=True)
    _gpu_name = _gpu.stdout.strip() if _gpu.returncode == 0 else ""
except FileNotFoundError:
    _gpu_name = ""
assert _gpu_name, (
    "❌ GPU yok — Runtime → Change runtime type → T4 GPU seç ve Run all'ı yeniden çalıştır"
)

# === xAI ===
XAI_MODEL = "grok-4.3"
XAI_URL   = "https://api.x.ai/v1/chat/completions"

def xai_probe(key, *, fatal):
    import requests
    try:
        answer = requests.post(
            XAI_URL,
            headers={"Authorization": f"Bearer {key}", "Content-Type": "application/json"},
            json={"model": XAI_MODEL,
                  "messages": [{"role": "user", "content": "ping"}],
                  "max_tokens": 1},
            timeout=30,
        )
    except Exception as unreachable:
        print(f"⚠️  xAI yoklanamadı ({type(unreachable).__name__}: {unreachable}) — "
              f"anahtar hakkında bir şey söylenemiyor, koşu sürüyor")
        return
    if answer.status_code // 100 == 2:
        print(f"✓ xAI anahtarı çalışıyor (model: {XAI_MODEL})")
        return
    said = (f"xAI anahtarı reddedildi — HTTP {answer.status_code}, "
            f"xAI yanıtı: {answer.text[:400]}")
    if fatal:
        raise RuntimeError(
            f"❌ {said}\nVideo kuruluyor ama video prompt'unu bu anahtar yazıyor, başka yolu yok. "
            f"console.x.ai'dan yeni anahtar al ve Colab Secrets'taki XAI_API_KEY'i güncelle."
        )
    print(f"⚠️  {said}")
    print("   Video üretmeyeceksen sorun değil — foto üretimi anahtarsız da çalışır.")

_chosen = [name for name, on in (("fotoğraf", INSTALL_PHOTO), (f"video ({VIDEO_MODEL})", INSTALL_VIDEO),
                                 ("ses", INSTALL_AUDIO)) if on]
print(f"✓ GPU: {_gpu_name}")
print("✓ CONFIG hazır (token Colab Secrets'tan okundu)")
print(f"✓ Kurulacak üretici: {', '.join(_chosen)}")
print(f"✓ Dal: {BRANCH}  |  Repo: {REPO}  |  Hedef: {CLONE_DIR}")
print(f"✓ Proje kökü: MyDrive/{DRIVE_FOLDER}")
print(f"✓ HF aynası: {HF_MIRROR}")
if not XAI_API_KEY:
    print("✓ xAI anahtarı: yok — video prompt yazılamaz (foto üretimi etkilenmez)")
else:
    xai_probe(XAI_API_KEY, fatal=INSTALL_VIDEO)

In [ ]:
# === Mount Google Drive ===
import os
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
os.makedirs(DRIVE_ROOT, exist_ok=True)
assert os.path.isdir(DRIVE_ROOT), f"❌ Proje kökü oluşmadı: {DRIVE_ROOT}"
print(f"✓ Drive bağlı — proje kökü: {DRIVE_ROOT}")

In [ ]:
# === Clone ===
import os, shutil, subprocess

def _mask(text):
    return text.replace(GITHUB_TOKEN, "<token>") if GITHUB_TOKEN else text

if os.path.exists(CLONE_DIR):
    shutil.rmtree(CLONE_DIR)

clone_url = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git"
result = subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--depth", "1", clone_url, CLONE_DIR],
    capture_output=True, text=True,
)
if result.returncode != 0:
    raise RuntimeError("❌ Klon başarısız:\n" + _mask(result.stderr.strip() or result.stdout.strip()))

DIST = os.path.join(CLONE_DIR, "queen-editor", "frontend", "dist", "index.html")
assert os.path.exists(DIST), f"❌ Derlenmiş arayüz yok: {DIST} — frontend'i derleyip commit'le (README)"

for _name in ("workflow_api.json", "workflow_video_api.json",
              "workflow_video_first_last_api.json", "workflow_video_h3_api.json",
              "workflow_video_h3_first_last_api.json"):
    _path = os.path.join(CLONE_DIR, "queen-editor", "assets", _name)
    assert os.path.exists(_path), f"❌ Grafik yok: {_path} — {_name} commit'lenmiş mi?"
print("✓ Klon tamam (derlenmiş arayüz + beş grafik mevcut)")

In [ ]:
# === Shared helpers ===
import sys

assert "COMFY_ROOT" in globals(), "❌ Önce 1) CONFIG hücresini çalıştır"

for _name in [n for n in sys.modules if n == "colab" or n.startswith("colab.")]:
    del sys.modules[_name]
if APP_DIR not in sys.path:
    sys.path.insert(0, APP_DIR)
from colab.console import log, human, run
from colab.downloads import fetch, hf_fetch, civitai_fetch, download_summary

print("✓ Ortak yardımcılar hazır (klondan: colab/console.py, colab/downloads.py)")

## ComfyUI + Custom Node'lar (21)

In [ ]:
%cd /content

# === System deps + ComfyUI ===
!apt-get install -y aria2 ffmpeg > /dev/null 2>&1
![ -d ComfyUI ] || git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!git pull -q
!pip install -q -r requirements.txt
!pip install -q opencv-python imageio imageio-ffmpeg

# === Custom nodes ===
import os
%cd /content/ComfyUI/custom_nodes

CUSTOM_NODES = [
    ("ComfyUI-Manager",           "https://github.com/ltdrdata/ComfyUI-Manager.git"),
    ("rgthree-comfy",             "https://github.com/rgthree/rgthree-comfy.git"),
    ("ComfyUI-Impact-Pack",       "https://github.com/ltdrdata/ComfyUI-Impact-Pack.git"),
    ("ComfyUI-Impact-Subpack",    "https://github.com/ltdrdata/ComfyUI-Impact-Subpack.git"),
    ("ComfyUI-Easy-Use",          "https://github.com/yolain/ComfyUI-Easy-Use.git"),
    ("ComfyUI-Custom-Scripts",    "https://github.com/pythongosssss/ComfyUI-Custom-Scripts.git"),
    ("ComfyUI_UltimateSDUpscale", "https://github.com/ssitu/ComfyUI_UltimateSDUpscale.git"),
    ("ComfyUI-KJNodes",           "https://github.com/kijai/ComfyUI-KJNodes.git"),
    ("ComfyUI-ppm",               "https://github.com/pamparamm/ComfyUI-ppm.git"),
    ("comfy_mtb",                 "https://github.com/melMass/comfy_mtb.git"),
    ("ComfyUI-VideoHelperSuite",  "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git"),
    ("ComfyUI-WanVideoWrapper",   "https://github.com/kijai/ComfyUI-WanVideoWrapper.git"),
    ("ComfyUI-GGUF",              "https://github.com/city96/ComfyUI-GGUF.git"),
    ("ComfyMath",                 "https://github.com/evanspearman/ComfyMath.git"),
    ("ComfyUI-Frame-Interpolation", "https://github.com/Fannovel16/ComfyUI-Frame-Interpolation.git"),
    ("ComfyUI-VFI",               "https://github.com/GACLove/ComfyUI-VFI.git"),
    ("ComfyUI_Comfyroll_CustomNodes", "https://github.com/Suzie1/ComfyUI_Comfyroll_CustomNodes.git"),
    ("ComfyUI-mxToolkit",         "https://github.com/Smirnov75/ComfyUI-mxToolkit.git"),
    ("ComfyUI-NAG",               "https://github.com/scottmudge/ComfyUI-NAG.git"),
    ("comfyui-adaptiveprompts",   "https://github.com/Alectriciti/comfyui-adaptiveprompts.git"),
    ("ComfyUI-DaSiWa-Nodes",      "https://github.com/darksidewalker/ComfyUI-DaSiWa-Nodes.git"),
]

for name, url in CUSTOM_NODES:
    if os.path.exists(name) and os.listdir(name):
        log(f"{name}: zaten var")
        continue
    log(f"{name}: cloning...")
    run(["git", "clone", "--depth", "1", "--recurse-submodules", url, name], f"clone {name}", timeout=180)
    if not os.listdir(name):
        raise RuntimeError(f"{name}: klon sonrası klasör boş")
    req = f"/content/ComfyUI/custom_nodes/{name}/requirements.txt"
    if os.path.exists(req):
        run(f"pip install -q -r {req}", f"pip install {name}", timeout=300)

log(f"{len(CUSTOM_NODES)} custom node hazır", "OK")

# === HF downloader ===
!pip install -q -U hf_xet

## Modeller — seçilen üreticiler, Civitai'dekiler önce HF aynasından

In [ ]:
import os, glob, shutil

# === Target folders ===
COMFY = COMFY_ROOT
CKPT = f"{COMFY}/models/checkpoints"
LORA = f"{COMFY}/models/loras"
UPSC = f"{COMFY}/models/upscale_models"
BBOX = f"{COMFY}/models/ultralytics/bbox"
SAMS = f"{COMFY}/models/sams"
DIFF = f"{COMFY}/models/diffusion_models"
VAE  = f"{COMFY}/models/vae"
TENC = f"{COMFY}/models/text_encoders"
CLIPV = f"{COMFY}/models/clip_vision"
MMAU = f"{COMFY}/models/mmaudio"
H3DIFF = f"{DIFF}/MiniMaxH3"
H3VAE  = f"{VAE}/MiniMaxH3"
TAE    = f"{COMFY}/models/vae_approx"
for d in [CKPT, LORA, UPSC, BBOX, SAMS, DIFF, VAE, TENC, CLIPV, MMAU, H3DIFF, H3VAE, TAE]:
    os.makedirs(d, exist_ok=True)

# === Photo ===
PHOTO_CHECKPOINTS = [
    (2744564, 7, "nova3DCGXL_ilV90.safetensors", "Nova 3DCG XL IL v9.0"),
    (3012006, 7, "DasiwaIllustriousAnime_epitaphecstasy.safetensors",
     "DaSiWa Illustrious Anime EpitaphEcstasy"),
]

PHOTO_MODELS = [
    (PHOTO_NOVA3DCG, "nova3dcg", "nova3DCGXL_ilV90.safetensors"),
    (PHOTO_DASIWA, "dasiwa", "DasiwaIllustriousAnime_epitaphecstasy.safetensors"),
]
CHOSEN_MODELS = [mid for on, mid, _fn in PHOTO_MODELS if on]
CHOSEN_CHECKPOINTS = [fn for on, _mid, fn in PHOTO_MODELS if on]

CIVITAI_PHOTO = [
    (1552087, LORA, "USNR_STYLE_ILL_V1_lokr3-000024.safetensors", "USNR STYLE ILL v1.0"),
    (3077575, LORA, "translucent_penetration_v5.safetensors", "Translucent Penetration v5"),
] + [(vid, CKPT, fn, label) for vid, _gib, fn, label in PHOTO_CHECKPOINTS
     if fn in CHOSEN_CHECKPOINTS]
HF_PHOTO = [
    ("FacehugmanIII/4x_foolhardy_Remacri", "4x_foolhardy_Remacri.pth",
     UPSC, "4x_foolhardy_Remacri.pth", "Remacri 4x upscaler", 50_000_000),
    ("Bingsu/adetailer", "face_yolov9c.pt",
     BBOX, "face_yolov9c.pt", "Yuz dedektoru (yolov9c)", 40_000_000),
]
OPEN_PHOTO = [
    ("https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth",
     SAMS, "sam_vit_b_01ec64.pth", "SAM ViT-B", 300_000_000),
]

# === Video — WAN ===
WAN22 = "Comfy-Org/Wan_2.2_ComfyUI_Repackaged"
WAN21 = "Comfy-Org/Wan_2.1_ComfyUI_repackaged"
HF_VIDEO = [
    (WAN22, "split_files/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors",
     LORA, "wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors", "Lightx2v I2V HIGH", None),
    (WAN22, "split_files/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors",
     LORA, "wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors", "Lightx2v I2V LOW", None),
    (WAN21, "split_files/vae/wan_2.1_vae.safetensors",
     VAE, "Wan2_1_VAE_fp32.safetensors", "Wan2.1 VAE", None),
    (WAN21, "split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
     TENC, "umt5_xxl_fp8_e4m3fn_scaled.safetensors", "UMT5-XXL", None),
    (WAN21, "split_files/clip_vision/clip_vision_h.safetensors",
     CLIPV, "clip_vision_h.safetensors", "CLIP Vision H", None),
]
CIVITAI_VIDEO = [
    (2513182, DIFF, "SmoothMix_I2V_v2_High.safetensors",         "SmoothMix I2V v2 HIGH"),
    (2513186, DIFF, "SmoothMix_I2V_v2_Low.safetensors",          "SmoothMix I2V v2 LOW"),
    (2376136, LORA, "SmoothMix_Animations_XXX_High.safetensors", "SmoothMix Animations XXX HIGH"),
    (2376143, LORA, "SmoothMix_Animations_XXX_Low.safetensors",  "SmoothMix Animations XXX LOW"),
]

# === Video — H3 ===
HF_H3 = [
    ("Abiray/MiniMax-H3-GGUF", "text_encoders/qwen3vl_32b_minimax_h3_int4_convrot.safetensors",
     TENC, "qwen3vl_32b_minimax_h3_int4_convrot.safetensors", "H3 Qwen3-VL", None),
    ("Kijai/MiniMax-H3-experimental", "minimax_h3_video_vae_int8_convrot.safetensors",
     H3VAE, "minimax_h3_video_vae_int8_convrot.safetensors", "H3 video VAE", None),
    ("Comfy-Org/MiniMax-H3", "vae/minimax_h3_audio_vae_fp32.safetensors",
     H3VAE, "minimax_h3_audio_vae_fp32.safetensors", "H3 ses VAE", None),
    ("Kijai/MiniMax-H3-TAE", "vae_approx/taeh3.safetensors",
     TAE, "taeh3.safetensors", "H3 TAE", None),
]
CIVITAI_H3 = [
    (3314686, H3DIFF,
     "dasiwa_minimax_h3_ref2va_v2_pruned_hybrid_turbo_int8_row-wise_convrot_runtime_mixed.safetensors",
     "DaSiWa H3 Hybrid Turbo v2"),
    (3228867, LORA, "H3_Motion_BoosterV2.safetensors", "H3 Motion Booster"),
]

# === Audio ===
HF_AUDIO = [
    ("phazei/NSFW_MMaudio", "mmaudio_large_44k_nsfw_gold_8.5k_final_fp16.safetensors",
     MMAU, "mmaudio_large_44k_nsfw_gold_8.5k_final_fp16.safetensors", "MMAudio NSFW fine-tune",
     None),
]

# === What this run installs ===
civitai_jobs = ((CIVITAI_PHOTO if INSTALL_PHOTO else [])
                + (CIVITAI_VIDEO if VIDEO_MODEL == "wan" else [])
                + (CIVITAI_H3 if VIDEO_MODEL == "h3" else []))
hf_jobs = ((HF_PHOTO if INSTALL_PHOTO else [])
           + (HF_VIDEO if VIDEO_MODEL == "wan" else [])
           + (HF_H3 if VIDEO_MODEL == "h3" else [])
           + (HF_AUDIO if INSTALL_AUDIO else []))
open_jobs = OPEN_PHOTO if INSTALL_PHOTO else []

# === Disk ===
PHOTO_GIB = 2 + sum(gib for _vid, gib, fn, _label in PHOTO_CHECKPOINTS
                    if fn in CHOSEN_CHECKPOINTS)
SIZES = [(INSTALL_PHOTO, PHOTO_GIB, "fotoğraf"), (VIDEO_MODEL == "wan", 39, "video (WAN)"),
         (VIDEO_MODEL == "h3", 37, "video (H3)"), (INSTALL_AUDIO, 9, "ses")]
HEADROOM_GIB = 5
need = sum(gib for on, gib, _ in SIZES if on)
free = shutil.disk_usage("/content").free / 1024**3
log(f"Seçim: {', '.join(name for on, _, name in SIZES if on)} — ~{need} GiB "
    f"| Diskte boş: {free:.1f} GiB")
if free < need + HEADROOM_GIB:
    raise RuntimeError(
        f"❌ Disk yetmiyor: ~{need} GiB model + {HEADROOM_GIB} GiB pay gerekiyor, "
        f"{free:.1f} GiB boş. Daha az üretici ya da daha az foto modeli seç, ya da diski daha "
        f"büyük bir runtime aç."
    )

# === Hugging Face downloads ===
landed = []
for repo, path, d, fn, label, floor in hf_jobs:
    landed.append(hf_fetch(repo, path, d, fn, label, floor=floor))

# === Open downloads ===
for url, d, fn, label, floor in open_jobs:
    landed.append(fetch(url, d, fn, label, parallel=True, floor=floor))

# === Civitai downloads ===
for vid, d, fn, label in civitai_jobs:
    landed.append(civitai_fetch(HF_MIRROR, vid, d, fn, label, COOKIE_VALUE))

# === Summary ===
folders = []
if INSTALL_PHOTO:
    folders += [("checkpoints", CKPT, "*.safetensors"), ("upscale_models", UPSC, "*.pth"),
                ("ultralytics/bbox", BBOX, "*.pt"), ("sams", SAMS, "*.pth")]
if INSTALL_VIDEO:
    folders += [("diffusion_models", DIFF, "**/*.safetensors"), ("vae", VAE, "**/*.safetensors"),
                ("text_encoders", TENC, "*.safetensors"), ("clip_vision", CLIPV, "*.safetensors"),
                ("vae_approx", TAE, "*.safetensors")]
if INSTALL_PHOTO or INSTALL_VIDEO:
    folders += [("loras", LORA, "*.safetensors")]
if INSTALL_AUDIO:
    folders += [("mmaudio", MMAU, "*.safetensors")]
for title, folder, pattern in folders:
    print(f"\n📂 {title}/")
    for f in sorted(glob.glob(f"{folder}/{pattern}", recursive=True)):
        print(f"   {human(os.path.getsize(f))}  {os.path.relpath(f, folder)}")
download_summary(landed)
if INSTALL_PHOTO:
    log(f"Fotoğraf modelleri: {', '.join(CHOSEN_MODELS)}")
log("Seçilen modeller indirildi ve doğrulandı", "OK")

## Ses motoru (yalnız `INSTALL_AUDIO` işaretliyse)

In [ ]:
# === Ses motoru — MMAudio kütüphanesi ===
import os

MMAUDIO_DIR = "/content/MMAudio"

if not INSTALL_AUDIO:
    log("Ses motoru: atlandı (INSTALL_AUDIO kapalı)")
else:
    if not os.path.isdir(MMAUDIO_DIR):
        run(["git", "clone", "--depth", "1", "https://github.com/hkchengrex/MMAudio.git",
             MMAUDIO_DIR], "clone MMAudio", timeout=300)
    run(["pip", "install", "-e", ".", "-q"], "pip install MMAudio", cwd=MMAUDIO_DIR, timeout=1800)
    log("MMAudio kütüphanesi kuruldu", "OK")

In [ ]:
# === Ses motoru — MMAudio'nun kendi ağırlıkları (~7 GiB) ===
import os, sys

if not INSTALL_AUDIO:
    log("MMAudio ağırlıkları: atlandı (INSTALL_AUDIO kapalı)")
else:
    assert os.path.isdir(APP_DIR), f"❌ Uygulama klasörü yok: {APP_DIR} — önce klon hücresini çalıştır"
    if MMAUDIO_DIR not in sys.path:
        sys.path.insert(0, MMAUDIO_DIR)
    _cwd = os.getcwd()
    os.chdir(APP_DIR)
    try:
        from mmaudio.eval_utils import all_model_cfg
        all_model_cfg["large_44k"].download_if_needed()
    finally:
        os.chdir(_cwd)
    log(f"MMAudio ağırlıkları hazır → {APP_DIR}", "OK")

## ComfyUI'yi başlat (arka planda)

In [ ]:
import subprocess, time, os, urllib.request

# === Start ComfyUI ===
os.system("pkill -f 'python main.py' 2>/dev/null")
time.sleep(2)

comfy_log = open(COMFY_LOG, "w")
proc = subprocess.Popen(
    ["python", "main.py", "--listen", "127.0.0.1", "--port", str(COMFY_PORT)],
    cwd=COMFY_ROOT, stdout=comfy_log, stderr=subprocess.STDOUT,
)
log(f"ComfyUI başlatıldı (PID {proc.pid}), log: {COMFY_LOG}")

# === Ready? ===
for i in range(45):
    time.sleep(2)
    try:
        urllib.request.urlopen(f"{COMFYUI_URL}/system_stats", timeout=2)
        log(f"ComfyUI hazır ({(i + 1) * 2}s)", "OK")
        break
    except Exception:
        pass
else:
    with open(COMFY_LOG) as f:
        print("".join(f.readlines()[-30:]))
    raise RuntimeError("❌ ComfyUI 90 sn içinde başlamadı — yukarıdaki log'a bak")

In [ ]:
# === Start Flask + cloudflared tunnel ===
import subprocess, time, os, re, urllib.request

FLASK_LOG = "/content/flask.log"

subprocess.run(["pkill", "-f", "backend.main"], check=False)
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
time.sleep(2)

logf = open(FLASK_LOG, "w")
flask_env = {**os.environ, "QE_DRIVE_ROOT": DRIVE_ROOT, "QE_COMFY_URL": COMFYUI_URL,
             "QE_COMFY_ROOT": COMFY_ROOT, "QE_COMFY_LOG": COMFY_LOG,
             "QE_PHOTO_MODELS": ",".join(CHOSEN_MODELS), "QE_VIDEO_MODEL": VIDEO_MODEL,
             "QE_XAI_API_KEY": XAI_API_KEY or "",
             "QE_XAI_MODEL": XAI_MODEL, "QE_XAI_URL": XAI_URL}
subprocess.Popen(["python", "-m", "backend.main"], cwd=APP_DIR, env=flask_env,
                 stdout=logf, stderr=subprocess.STDOUT)

ok = False
for i in range(45):
    time.sleep(2)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{APP_PORT}/api/health", timeout=2)
        ok = True
        break
    except Exception:
        pass
if not ok:
    print("".join(open(FLASK_LOG).readlines()[-30:]))
    raise RuntimeError("❌ Flask 90 sn içinde /api/health'e cevap vermedi — yukarıdaki log'a bak")
print(f"✓ Flask ayakta ({(i + 1) * 2}s)")

if not os.path.isfile("/content/cloudflared"):
    subprocess.run(["wget", "-q", "-O", "/content/cloudflared",
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    subprocess.run(["chmod", "+x", "/content/cloudflared"], check=True)

tunlog = "/content/cloudflared.log"
subprocess.Popen(["/content/cloudflared", "tunnel", "--protocol", "http2",
                  "--url", f"http://127.0.0.1:{APP_PORT}"],
                 stdout=open(tunlog, "w"), stderr=subprocess.STDOUT)
link = None
for _ in range(30):
    time.sleep(1)
    if os.path.exists(tunlog):
        m = re.search(r"https://[-\w.]+trycloudflare\.com", open(tunlog).read())
        if m:
            link = m.group(0)
            break
if not link:
    print(open(tunlog).read()[-1000:] if os.path.exists(tunlog) else "(cloudflared log yok)")
    raise RuntimeError("❌ cloudflared linki 30 sn içinde alınamadı")

print(f"✓ Link {cell_elapsed()}'de hazır")
print(f"\n🔗 Queen Editor: {link}\n")
print("⬆️  Linke gir → projeye tıkla → prompt yaz → Üret.\n")
print("📡 Sunucu çalışıyor — BU HÜCREYİ KAPATMA. Canlı log:\n")
try:
    subprocess.run(["tail", "-n", "+1", "-f", FLASK_LOG])
except KeyboardInterrupt:
    print("Hücre durduruldu — Flask hâlâ arka planda (yeni link için tekrar çalıştır).")